---
title: Alignment Metrics
subtitle: A view of alignment performance
date: "9999-12-31"
edit_url: null
---

**Linked-read type**: PLACEHOLDER

In [ ]:
import altair as alt
from harpy.report.components import ITable, StatsBox, SafeRender, print_html
import os
import polars as pl
pl.Config.set_tbl_hide_column_data_types()



In [ ]:
indir = "reports/data"
platform = "haplotagging"


In [ ]:
keys_to_keep = set([
    "percentage_of_properly_paired_reads_(%)",
    "pairs_on_different_chromosomes",
    "reads_mapped_%",
#    "reads_unmapped_%",
    "reads_QC_failed_%",
    "error_rate",
    "supplementary_alignments",
    "raw_total_sequences",
    "insert_size_average",
    'self_&_mate_mapped_%'
])

In [ ]:
def get_val(x: str):
    '''Return the value of the line `x` as an int'''
    return int(x.strip().split(": ")[-1])

def parse_markdup(filename):
    '''Parses a samtools markdup output log file and returns the percent of duplicates as a tuple (pcr, optical)'''
    pcr = 0
    optical = 0
    reads = 0
    with open(filename, 'r') as f:
        for line in f:
            if line.startswith("READ:"):
                reads = get_val(line)
            if line.startswith("DUPLICATE PAIR:") or line.startswith("DUPLICATE SINGLE:"):
                pcr += get_val(line)
            if line.startswith("DUPLICATE PAIR OPTICAL:") or line.startswith("DUPLICATE SINGLE OPTICAL:"):
                optical += get_val(line)
    pcr_perc = round(((pcr-optical) / reads) * 100, 2)
    optical_perc = round((optical / reads) * 100, 2)
    return pcr_perc, optical_perc

def parse_SN(filecontents: list[str]):
    """Parse `SN` rows from samtools stats output into a normalized dict.
    Also extract samtools and htslib versions if present on the header line.
    Returns a tuple: (parsed_data, samtools_version, htslib_version)
    """
    parsed_data = {}
    for line in filecontents:
        if not line.startswith("SN"):
            if line.startswith("# First Fragment Qualities."):
                break
            continue
        sections = line.split("\t")
        if len(sections) < 3:
            continue
        field = sections[1].strip()[:-1].replace(" ", "_").lstrip()
        try:
            value = float(sections[2].strip())
        except ValueError:
            continue
        parsed_data[field] = value
    return parsed_data


In [ ]:
def parse_samtools_stats(indir: str):
    """Find Samtools stats logs and parse their data"""
    statsdir = os.path.join(indir, "samtools_stats")
    markdupdir = os.path.join(indir, "markdup")
    samtools_stats = {}
    for f in os.listdir(statsdir):
        samplename = os.path.basename(f).replace(".stats", "")
        with open(os.path.join(statsdir, f), 'r') as f:
            file_contents = f.read().splitlines()
        parsed_data = parse_SN(file_contents)
        pcr, optical = parse_markdup(os.path.join(markdupdir, f"{samplename}.markdup"))

        if len(parsed_data) > 0:
            # Work out some percentages
            if "raw_total_sequences" in parsed_data:
                parsed_data['self_&_mate_mapped_%'] = round(parsed_data['reads_mapped_and_paired'] / parsed_data["sequences"] * 100, 1)
                parsed_data['error_rate'] = round(parsed_data['error_rate'] * 100, 4)
                for k in list(parsed_data.keys()):
                    if k.startswith("reads_") and k != "raw_total_sequences" and parsed_data["raw_total_sequences"] > 0:
                        parsed_data[f"{k}_%"] = round((parsed_data[k] / parsed_data["raw_total_sequences"]) * 100, 1)
                        del parsed_data[k]
            samtools_stats[samplename] = {key.replace("_"," ").replace(" and ", " & ").replace("reads", "").title().strip(): parsed_data[key] for key in keys_to_keep}
            samtools_stats[samplename]["% Duplicates (PCR)"] = pcr
            samtools_stats[samplename]["% Duplicates (Opt)"] = optical
    if len(samtools_stats) == 0:
        return {}
    return samtools_stats


In [ ]:
dat = parse_samtools_stats(indir)


In [ ]:
df = pl.DataFrame([{"Sample": sample, **cols} for sample, cols in dat.items()]).rename({
    #"Unmapped %": "% Unmapped",
    "Self & Mate Mapped %": "% Self+Mate Mapped",
    "Qc Failed %": "% QC Failed",
    "Raw Total Sequences": "Reads Mapped",
    "Insert Size Average": "Average Insert (bp)",
    "Percentage Of Properly Paired  (%)": "% Properly Paired",
    "Pairs On Different Chromosomes": "Diff Chromosome",
    "Mapped %": "% Mapped",
    "Error Rate": "Error Rate (%)",
    "Supplementary Alignments": "Supp. Alignments"    
}).select([
    "Sample",
    "Reads Mapped",
    "% Mapped",
    "% Properly Paired",
    "% Self+Mate Mapped",
    "% Duplicates (PCR)",
    "% Duplicates (Opt)",
    "Average Insert (bp)",
    "% QC Failed",
    "Diff Chromosome",
    "Error Rate (%)",
    "Supp. Alignments",
])



In [ ]:
_pp = df["% Properly Paired"].replace(0, None).mean() or 0
_ppstd = df["% Properly Paired"].replace(0, None).std() or 0

_smm = df["% Self+Mate Mapped"].replace(0, None).mean() or 0
_smmstd = df["% Self+Mate Mapped"].replace(0, None).std() or 0

_pcr = df["% Duplicates (PCR)"].mean() or 0
_pcrstd = df["% Duplicates (PCR)"].std() or 0

_opt = df["% Duplicates (Opt)"].mean() or 0
_optstd = df["% Duplicates (Opt)"].std() or 0

_isize = round(df['Average Insert (bp)'].replace(0, None).mean() or 0)
_isizestd = round(df['Average Insert (bp)'].replace(0, None).std() or 0)

_err = df["Error Rate (%)"].replace(0, None).mean() or 0
_errstd = df["Error Rate (%)"].replace(0, None).std() or 0

(
    StatsBox()
    .add(len(df), "Samples")
    .conditional(_pp, "Properly Paired", 80, plus_minus=_ppstd, add_percent=True, digits = 1)
    .conditional(_smm, "Self+Mate Mapped", 80, plus_minus=_smmstd, add_percent=True, digits = 1)
    .conditional(_pcr, "PCR Duplicates", 40, plus_minus=_pcrstd, lower_bad = False, add_percent=True, digits = 2)
    .conditional(_opt, "Optical Duplicates", 40, plus_minus=_optstd, lower_bad = False, add_percent=True, digits = 2)
    .conditional(_err, "Error Rate", 10, plus_minus=_errstd, lower_bad = False, add_percent=True, digits = 3)
    .add(_isize, "Insert Size", plus_minus=_isizestd, units = "bp")
).render()

In [ ]:
ITable(df, filename = "Align.stats", fixedcols = 1).render()